# app_stats demo

End-to-end walkthrough: seed the DB → serve the router → fetch via `StatsApiClient` → ingest → plot.

Run inside the dev container (`make launch-jupyter`) where Postgres and ecodev_core are already wired.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import plotly.express as px
from fastapi import FastAPI
from fastapi.testclient import TestClient
from sqlmodel import Session

from ecodev_core import (
    AppActivity, create_db_and_tables, delete_table, engine,
    get_stats_router, ProjectExport, ProjectStatsAdapter,
)
from ecodev_core.app_stats.consumer import (
    StatsApiClient,
    RemoteHourlyActivity, RemoteAppProject,
    delete_lookback_activities, delete_lookback_projects,
    upsert_remote_activities, upsert_remote_projects,
    get_activities_df, get_projects_df,
)

## 1. Seed the database

In [ ]:
SEED_FILE = Path('/app/tests/fixtures/app_stats_seed.json')
seed = json.loads(SEED_FILE.read_text())

create_db_and_tables(AppActivity)
delete_table(AppActivity)

with Session(engine) as session:
    for row in seed['activities']:
        session.add(AppActivity(
            user=row['user'],
            application=row['application'],
            method=row['method'],
            relevant_option=row.get('relevant_option'),
            created_at=datetime.fromisoformat(row['created_at']),
        ))
    session.commit()

print(f'Seeded {len(seed["activities"])} activity rows')

## 2. Serve the router (activities-only producer)

In [ ]:
TEST_API_KEY = 'demo-key'

import ecodev_core.app_stats.api_key as api_key_module
api_key_module._configured_api_key = lambda: TEST_API_KEY

app_producer = FastAPI()
app_producer.include_router(get_stats_router())
producer_client = TestClient(app_producer)

resp = producer_client.get('/stats/activities', headers={'X-API-Key': TEST_API_KEY})
print(f'Status: {resp.status_code}')
print(json.dumps(resp.json(), indent=2, default=str))

## 3. Serve with projects adapter (cf-tool style)

In [ ]:
def list_projects(session, from_date, to_date):
    return [ProjectExport(**p) for p in seed['projects']]

app_with_projects = FastAPI()
app_with_projects.include_router(
    get_stats_router(adapter=ProjectStatsAdapter(list_projects=list_projects))
)
projects_client = TestClient(app_with_projects)

resp = projects_client.get('/stats/projects', headers={'X-API-Key': TEST_API_KEY})
print(json.dumps(resp.json(), indent=2, default=str))

## 4. Fetch via StatsApiClient and ingest

In [ ]:
class _TestClientTransport:
    """Wraps FastAPI TestClient so StatsApiClient can drive it without a real network."""
    def __init__(self, test_client):
        self._tc = test_client

    def get(self, url, headers=None, params=None, timeout=None):
        from urllib.parse import urlencode, urlparse
        parsed = urlparse(url)
        path = parsed.path
        return self._tc.get(path, headers=headers, params=params)


create_db_and_tables(RemoteHourlyActivity)
create_db_and_tables(RemoteAppProject)
delete_table(RemoteHourlyActivity)
delete_table(RemoteAppProject)

client = StatsApiClient(base_url='http://testserver', api_key=TEST_API_KEY)

activities = list(client.fetch_activities.__wrapped__(client))

with Session(engine) as session:
    lookback = datetime(2026, 1, 1)
    delete_lookback_activities(session, 'my_ecoact', lookback)
    upsert_remote_activities(session, 'my_ecoact', activities)
    df_activities = get_activities_df(session)

print(f'Ingested {len(df_activities)} rows')
df_activities.head()

## 5. Monthly unique users chart

In [ ]:
with Session(engine) as session:
    df = get_activities_df(session)

df['month'] = pd.to_datetime(df['hour']).dt.to_period('M').astype(str)
monthly = df.groupby('month')['user_email'].nunique().reset_index()
monthly.columns = ['month', 'unique_users']

fig = px.bar(monthly, x='month', y='unique_users',
             title='Monthly Unique Users', labels={'unique_users': 'Users'})
fig.show()

## 6. Per-application usage chart

In [ ]:
with Session(engine) as session:
    df = get_activities_df(session)

app_usage = df.groupby('application')['activity_count'].sum().reset_index()
app_usage.columns = ['application', 'total_actions']

fig = px.bar(app_usage, x='application', y='total_actions',
             title='Total Actions per Application')
fig.show()

## 7. Project inventory table

In [ ]:
with Session(engine) as session:
    df_p = get_projects_df(session)

if df_p.empty:
    projects = [ProjectExport(**p) for p in seed['projects']]
    with Session(engine) as session:
        delete_lookback_projects(session, 'cf_tool')
        upsert_remote_projects(session, 'cf_tool', projects)
        df_p = get_projects_df(session)

df_p[['application', 'project_id', 'name', 'creator', 'project_type', 'client']]